## 🎯 Learning Objectives
* Understand the fundamental architecture of sequence-to-sequence (Seq2Seq) models.
* Explain the roles of the encoder and decoder components in a Seq2Seq model.
* Identify common applications of Seq2Seq models in Natural Language Processing.
* Implement a basic Seq2Seq model using PyTorch for a simple sequence transformation task.
* Analyze the limitations of vanilla Seq2Seq models and recognize the need for attention mechanisms.


## Sequence-to-Sequence Models: Understanding the Core of Modern NLP

Welcome to the fascinating world of Sequence-to-Sequence (Seq2Seq) models! These architectures are the bedrock of many advanced Natural Language Processing (NLP) applications, from machine translation to text summarization and even sophisticated chatbots. At its heart, a Seq2Seq model is designed to transform an input sequence into an output sequence, where the lengths of the input and output sequences can be different.

### The Analogy: A Translator and a Writer

Imagine you have a complex sentence in English that needs to be translated into French. You can think of a Seq2Seq model as having two main parts working in tandem:

1.  **The Translator (Encoder):** This component reads the entire English sentence, word by word, and tries to understand its full meaning and context. It compresses all this information into a fixed-size "thought vector" or "context vector." This vector is like a summary of the input sentence, capturing its essence without retaining the original word order explicitly.

2.  **The Writer (Decoder):** Once the Translator has finished its job and produced the thought vector, the Writer takes over. It receives this thought vector and, based on the summarized meaning, starts generating the French sentence, word by word. Crucially, the Writer also considers the words it has *already generated* to decide what word to produce next, ensuring grammatical correctness and coherence in the output language.

### The Encoder-Decoder Architecture

In technical terms, a Seq2Seq model typically consists of two recurrent neural networks (RNNs), often Long Short-Term Memory (LSTM) or Gated Recurrent Unit (GRU) networks:

*   **Encoder:** This RNN processes the input sequence element by element. After processing the entire input sequence, its final hidden state (or a combination of hidden states) becomes the context vector. This context vector is intended to encapsulate the information of the entire input sequence.

*   **Decoder:** This RNN takes the context vector from the encoder as its initial hidden state. It then generates the output sequence element by element. At each step, it takes the previously generated output element (or a special "start-of-sequence" token for the first step) and its current hidden state to predict the next element in the output sequence. This process continues until an "end-of-sequence" token is generated.

### The Challenge and the Solution: Attention

A major limitation of the original Seq2Seq architecture is that the entire input sequence must be compressed into a single fixed-size context vector. For long sequences, this can lead to an information bottleneck, where the decoder struggles to recall relevant parts of the input, especially those processed early by the encoder. This is akin to our translator trying to remember a very long English paragraph and summarize it into a single thought, then write a French paragraph from just that single thought.

This limitation was largely overcome by the introduction of **Attention Mechanisms**. Instead of a single context vector, attention allows the decoder to "look back" at different parts of the input sequence at each decoding step, dynamically weighting their importance. This significantly improves performance on longer sequences and is a cornerstone of modern NLP models, including the Transformer architecture which we will explore in later lessons.

### Why are Seq2Seq models important in 2026?

While Transformers have largely superseded vanilla RNN-based Seq2Seq models for state-of-the-art performance, understanding the Seq2Seq paradigm is crucial. Transformers themselves are built upon the encoder-decoder structure and heavily leverage attention mechanisms, which were first popularized within the Seq2Seq framework. Many practical applications still use or are inspired by Seq2Seq, and it provides a foundational understanding for more complex architectures. Think of it as learning the internal combustion engine before diving into electric vehicles – the principles are fundamental.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- 1. Define the Encoder --- 
# The Encoder processes the input sequence and produces a context vector.
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        
        # Embedding layer to convert input tokens (integers) into dense vectors
        self.embedding = nn.Embedding(input_dim, emb_dim)
        
        # GRU (Gated Recurrent Unit) as the recurrent layer
        # GRU is chosen for its efficiency and ability to capture long-range dependencies
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        # src = [batch size, src len]
        
        # Convert input tokens to embeddings
        embedded = self.dropout(self.embedding(src))
        # embedded = [batch size, src len, emb dim]
        
        # Pass embeddings through the GRU
        # output: all hidden states for each time step
        # hidden: final hidden state for each layer (context vector)
        output, hidden = self.rnn(embedded)
        # output = [batch size, src len, hid dim * n directions] (if bidirectional)
        # hidden = [n layers * n directions, batch size, hid dim]
        
        # For a simple unidirectional encoder, we typically use the final hidden state
        # as the context vector for the decoder.
        return hidden

# --- 2. Define the Decoder --- 
# The Decoder takes the context vector and generates the output sequence.
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        
        self.embedding = nn.Embedding(output_dim, emb_dim)
        
        # GRU for the decoder. It takes the context vector as initial hidden state.
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        
        # Linear layer to map decoder's output hidden state to vocabulary size
        self.fc_out = nn.Linear(hid_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden):
        # input = [batch size, 1] (single token at a time)
        # hidden = [n layers * n directions, batch size, hid dim] (context vector from encoder)
        
        # Convert input token to embedding
        embedded = self.dropout(self.embedding(input))
        # embedded = [batch size, 1, emb dim]
        
        # Pass embedding and previous hidden state through the GRU
        output, hidden = self.rnn(embedded, hidden)
        # output = [batch size, 1, hid dim]
        # hidden = [n layers * n directions, batch size, hid dim]
        
        # Predict the next token using the linear layer
        prediction = self.fc_out(output.squeeze(1))
        # prediction = [batch size, output dim]
        
        return prediction, hidden

# --- 3. Define the Seq2Seq Model --- 
# Combines the Encoder and Decoder.
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
        # Ensure encoder and decoder hidden dimensions and layers match
        assert encoder.hid_dim == decoder.hid_dim, \
            "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, \
            "Number of layers of encoder and decoder must be equal!"
            
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src = [batch size, src len]
        # trg = [batch size, trg len]
        # teacher_forcing_ratio is probability to use actual target output as next input
        
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        # Encode the source sequence to get the context vector
        hidden = self.encoder(src)
        
        # First input to the decoder is the <sos> (start-of-sequence) token
        # We assume the first token of the target sequence is the <sos> token
        input = trg[:, 0].unsqueeze(1) # [batch size, 1]
        
        for t in range(1, trg_len):
            # Decode one step at a time
            output, hidden = self.decoder(input, hidden)
            
            # Store prediction for this time step
            outputs[:, t, :] = output
            
            # Decide whether to use teacher forcing
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            
            # Get the highest predicted token from the model output
            top1 = output.argmax(1)
            
            # If teacher forcing, use actual next token; else, use model's prediction
            input = trg[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)
            
        return outputs

# --- 4. Prepare Dummy Data and Vocabulary --- 
# For demonstration, we'll create a very simple character-level translation task:
# Input: 'abc' -> Output: 'xyz'
# Input: 'def' -> Output: 'uvw'

# Define a simple vocabulary mapping characters to integers
vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, 
         'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8,
         'x': 9, 'y': 10, 'z': 11, 'u': 12, 'v': 13, 'w': 14}

# Reverse mapping for decoding
idx_to_char = {v: k for k, v in vocab.items()}

# Example sequences (padded to max length)
# Max input length = 3, Max output length = 3 (excluding <sos>/<eos>)
# Let's make it 5 for output to include <sos> and <eos>

def tokenize_and_pad(sentence, vocab, max_len):
    tokens = [vocab['<sos>']] + [vocab[char] for char in sentence] + [vocab['<eos>']]
    padded_tokens = tokens + [vocab['<pad>']] * (max_len - len(tokens))
    return padded_tokens[:max_len]

SRC_MAX_LEN = 5 # <sos> a b c <eos>
TRG_MAX_LEN = 5 # <sos> x y z <eos>

src_sentences = ['abc', 'def']
trg_sentences = ['xyz', 'uvw']

src_data = torch.LongTensor([tokenize_and_pad(s, vocab, SRC_MAX_LEN) for s in src_sentences])
trg_data = torch.LongTensor([tokenize_and_pad(t, vocab, TRG_MAX_LEN) for t in trg_sentences])

# --- 5. Model Instantiation and Training Setup --- 

INPUT_DIM = len(vocab) # Total vocabulary size
OUTPUT_DIM = len(vocab)
EMB_DIM = 128
HID_DIM = 256
N_LAYERS = 2
DROPOUT = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
# We ignore padding tokens in the loss calculation
TRG_PAD_IDX = vocab['<pad>']
criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)

# --- 6. Training Loop (Simplified) --- 

def train(model, src, trg, optimizer, criterion, clip):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    output = model(src, trg)
    
    # output = [batch size, trg len, output dim]
    # trg = [batch size, trg len]
    
    # Reshape for loss calculation: flatten output and target
    # Exclude the <sos> token from target for loss calculation
    output_dim = output.shape[-1]
    output = output[:, 1:].reshape(-1, output_dim) # Exclude <sos> token from output
    trg = trg[:, 1:].reshape(-1) # Exclude <sos> token from target
    
    # output = [(batch size - 1) * trg len, output dim]
    # trg = [(batch size - 1) * trg len]
    
    loss = criterion(output, trg)
    loss.backward()
    
    # Clip gradients to prevent exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
    
    optimizer.step()
    
    return loss.item()

N_EPOCHS = 100
CLIP = 1.0

print(f"Training on {device}")
for epoch in range(N_EPOCHS):
    loss = train(model, src_data.to(device), trg_data.to(device), optimizer, criterion, CLIP)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch: {epoch+1:03} | Loss: {loss:.4f}')

print("\nTraining complete!")

# --- 7. Inference (Translation) --- 

def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_len=5):
    model.eval()
    
    # Tokenize and numericalize input sentence
    tokens = [src_vocab['<sos>']] + [src_vocab[char] for char in sentence] + [src_vocab['<eos>']]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device) # Add batch dimension
    
    # Encode the source sentence
    with torch.no_grad():
        hidden = model.encoder(src_tensor)
        
    # Initialize target sequence with <sos> token
    trg_input = torch.LongTensor([trg_vocab['<sos>']]).unsqueeze(0).to(device)
    
    translated_tokens = []
    
    for i in range(max_len):
        with torch.no_grad():
            output, hidden = model.decoder(trg_input, hidden)
        
        # Get the predicted next token
        pred_token = output.argmax(1).item()
        translated_tokens.append(pred_token)
        
        # If <eos> token is predicted, stop decoding
        if pred_token == trg_vocab['<eos>']:
            break
            
        # Use the predicted token as the next input to the decoder
        trg_input = torch.LongTensor([pred_token]).unsqueeze(0).to(device)
        
    # Convert numerical tokens back to characters
    translated_chars = [idx_to_char[t] for t in translated_tokens]
    
    # Remove <sos> and <eos> tokens for final output
    if translated_chars[0] == '<sos>':
        translated_chars = translated_chars[1:]
    if '<eos>' in translated_chars:
        translated_chars = translated_chars[:translated_chars.index('<eos>')]
        
    return ''.join(translated_chars)

# Test the model
input_sentence_1 = 'abc'
output_translation_1 = translate_sentence(input_sentence_1, vocab, vocab, model, device, TRG_MAX_LEN)
print(f"Input: '{input_sentence_1}' -> Predicted Output: '{output_translation_1}'")

input_sentence_2 = 'def'
output_translation_2 = translate_sentence(input_sentence_2, vocab, vocab, model, device, TRG_MAX_LEN)
print(f"Input: '{input_sentence_2}' -> Predicted Output: '{output_translation_2}'")

# Example with a slightly different input (might not work well due to limited training data)
input_sentence_3 = 'a'
output_translation_3 = translate_sentence(input_sentence_3, vocab, vocab, model, device, TRG_MAX_LEN)
print(f"Input: '{input_sentence_3}' -> Predicted Output: '{output_translation_3}'")


### Interpreting the Code Output and Performance Trade-offs

The code above demonstrates a minimalistic, character-level Seq2Seq model. After running the training loop, you should observe the `Loss` value decreasing over epochs, indicating that the model is learning to map input sequences to target sequences. The final inference step shows the model attempting to translate the input characters. For our simple example, `abc` should ideally translate to `xyz` and `def` to `uvw`.

**Key Observations and Interpretations:**

*   **Loss Reduction:** A decreasing loss signifies that the model's predictions are getting closer to the actual target sequences. In our case, the `CrossEntropyLoss` measures how well the model predicts the next character in the target sequence.
*   **Teacher Forcing:** The `teacher_forcing_ratio` parameter is crucial during training. When `teacher_forcing_ratio` is 1.0, the decoder always receives the *correct* previous target token as input for the next step. This helps the model learn faster. When it's 0.0, the decoder always uses its *own prediction* as the next input, which can lead to exposure bias (the model never sees its own errors during training). A common practice is to anneal (gradually decrease) the teacher forcing ratio during training, allowing the model to become more robust.
*   **Simplified Example:** This implementation is highly simplified. Real-world Seq2Seq models for tasks like machine translation involve much larger vocabularies, longer sequences, and more complex data preprocessing. The character-level task here serves to illustrate the architecture without overwhelming complexity.

**Performance Trade-offs and Limitations:**

1.  **Information Bottleneck:** As discussed, the fixed-size context vector from the encoder is a major limitation. For longer input sequences, it becomes increasingly difficult for this single vector to encapsulate all necessary information, leading to degraded performance. This is where **attention mechanisms** become indispensable, allowing the decoder to selectively focus on relevant parts of the input at each decoding step.
2.  **Vanishing/Exploding Gradients:** While GRUs (and LSTMs) mitigate this problem compared to vanilla RNNs, deep recurrent networks can still suffer from vanishing or exploding gradients, making it hard to train them effectively on very long sequences.
3.  **Sequential Processing:** RNNs process sequences step-by-step, which inherently limits parallelization during training. This makes them slower to train on large datasets compared to architectures that can process all input elements simultaneously.
4.  **Lack of Bidirectional Context in Decoder:** Our simple decoder is unidirectional. In practice, decoders often benefit from looking at both past and future context (though this is more complex to implement in a purely auto-regressive generation setting).

### Typical Use Cases in 2026

Despite the rise of Transformers, the Seq2Seq paradigm remains fundamental. Here are some common applications:

*   **Machine Translation:** The classic application, translating text from one language to another (e.g., Google Translate).
*   **Text Summarization:** Condensing long documents into shorter, coherent summaries.
*   **Dialogue Systems/Chatbots:** Generating appropriate responses in conversational AI.
*   **Code Generation:** Translating natural language descriptions into programming code.
*   **Image Captioning:** Describing the content of an image in natural language (where the encoder processes the image and the decoder generates text).
*   **Speech Recognition:** Converting audio signals into text (encoder processes audio, decoder generates text).

Modern implementations of these tasks almost universally leverage the **Transformer architecture**, which is an evolution of the Seq2Seq model that replaces recurrent layers with self-attention mechanisms, enabling superior performance and parallelization. However, the core idea of an encoder processing an input sequence and a decoder generating an output sequence remains the guiding principle.


### Resources for Further Learning

*   **Original Sequence to Sequence Paper:** [Sequence to Sequence Learning with Neural Networks](https://arxiv.org/abs/1409.3215) by Sutskever et al. (2014)
*   **Attention Mechanism Paper:** [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473) by Bahdanau et al. (2014)
*   **PyTorch `nn.GRU` Documentation:** [torch.nn.GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html)
*   **PyTorch `nn.Embedding` Documentation:** [torch.nn.Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
*   **Hugging Face Transformers Library:** The go-to library for state-of-the-art NLP models, many of which are based on the encoder-decoder (Seq2Seq) paradigm with attention. Explore their [documentation](https://huggingface.co/docs/transformers/index) and [models](https://huggingface.co/models).
*   **Google AI Studio / Gemini API:** For practical application of advanced Seq2Seq-like models, explore the [Google AI Studio](https://ai.google.dev/) and its [Gemini API](https://ai.google.dev/docs/gemini_api_overview) for powerful text generation and translation capabilities.
